In [1]:
# Importation des bibliothèques
import pandas as pd
import numpy as np
import os
import joblib
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import warnings
from Orchestrator import DOSSIER_RACINE, DOSSIER_TRANSFORMED, DOSSIER_CLEANED
warnings.filterwarnings('ignore')


In [2]:
df_train = pd.read_csv(os.path.join(DOSSIER_CLEANED, 'train.csv'), sep=';')
df_val = pd.read_csv(os.path.join(DOSSIER_CLEANED, 'val.csv'), sep=';')
df_test = pd.read_csv(os.path.join(DOSSIER_CLEANED, 'test.csv'), sep=';')

cible = 'Default'

y_train = df_train[cible].copy()
y_val = df_val[cible].copy()
y_test = df_test[cible].copy()

X_train = df_train.drop(columns=[cible])
X_val = df_val.drop(columns=[cible])
X_test = df_test.drop(columns=[cible])

print(f"Chargement effectue. Dimensions initiales X_train : {X_train.shape}")

Chargement effectue. Dimensions initiales X_train : (178735, 16)


In [3]:
#Identification et encodage des variables catégorielles
colonnes_categorielles = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

encodeur = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encodeur.fit(X_train[colonnes_categorielles])

noms_colonnes_encodees = encodeur.get_feature_names_out(colonnes_categorielles)

X_train_cat = pd.DataFrame(encodeur.transform(X_train[colonnes_categorielles]), columns=noms_colonnes_encodees, index=X_train.index)
X_val_cat = pd.DataFrame(encodeur.transform(X_val[colonnes_categorielles]), columns=noms_colonnes_encodees, index=X_val.index)
X_test_cat = pd.DataFrame(encodeur.transform(X_test[colonnes_categorielles]), columns=noms_colonnes_encodees, index=X_test.index)

print(f"Encodage termine. Nombre de variables categorielles creees : {len(noms_colonnes_encodees)}")

Encodage termine. Nombre de variables categorielles creees : 22


In [4]:
#Mise à l'échelle des variables continues
colonnes_continues = X_train.select_dtypes(include=['float64', 'int64']).columns.tolist()

standardiseur = StandardScaler()
standardiseur.fit(X_train[colonnes_continues])

X_train_num = pd.DataFrame(standardiseur.transform(X_train[colonnes_continues]), columns=colonnes_continues, index=X_train.index)
X_val_num = pd.DataFrame(standardiseur.transform(X_val[colonnes_continues]), columns=colonnes_continues, index=X_val.index)
X_test_num = pd.DataFrame(standardiseur.transform(X_test[colonnes_continues]), columns=colonnes_continues, index=X_test.index)

print("Standardisation terminee.")

Standardisation terminee.


In [5]:
#Recombinaison et exportation des artefacts
X_train_final = pd.concat([X_train_num, X_train_cat], axis=1)
X_val_final = pd.concat([X_val_num, X_val_cat], axis=1)
X_test_final = pd.concat([X_test_num, X_test_cat], axis=1)

df_train_final = pd.concat([X_train_final, y_train], axis=1)
df_val_final = pd.concat([X_val_final, y_val], axis=1)
df_test_final = pd.concat([X_test_final, y_test], axis=1)

dossier_export = os.path.join(DOSSIER_RACINE, 'data_transformed')
os.makedirs(dossier_export, exist_ok=True)

df_train_final.to_csv(os.path.join(dossier_export, 'train_transformed.csv'), index=False, sep=';')
df_val_final.to_csv(os.path.join(dossier_export, 'val_transformed.csv'), index=False, sep=';')
df_test_final.to_csv(os.path.join(dossier_export, 'test_transformed.csv'), index=False, sep=';')

joblib.dump(encodeur, os.path.join(dossier_export, 'encodeur.joblib'))
joblib.dump(standardiseur, os.path.join(dossier_export, 'standardiseur.joblib'))

print(f"Exportation terminee dans le repertoire : {dossier_export}")
print(f"Nouvelles dimensions du jeu d'entrainement : {df_train_final.shape}")

Exportation terminee dans le repertoire : RN_sousdoss/data_transformed
Nouvelles dimensions du jeu d'entrainement : (178735, 32)
